# Using the singlet excitation functions for MOPAC calculations

In this tutorial, we demonstrate the use of the more efficient singlet excitation specific functions for calculating energies, orbitals, time-overlaps, etc.

Strucurally, the workflow remains the same as the other tutorials but different underlying functions are used. This tutorial will focus on the different functions and only briefly go over the surrounding workflow. For a detailed overview of the workflow see 2_improved_tutorial.

## Table of contents
<a name="toc"></a>
1. [Importing needed libraries](#1)
2. [Reading a snapshot from the xyz trajectory file](#2)
3. [Creating input files for MOPAC calculations](#3)
4. [Running MOPAC calculations](#4)
5. [Reading molecular orbitals/energies and CI vectors](#5)
6. [Computing time-overlaps](#6)
7. [Reading molecular orbitals/energies and CI vectors. Using active space of MOs](#7)
8. [Possible sources of errors](#8)
9. [The remedy](#9)

   9.1. [Major flaw remedy - but low accuracy](#9.1)
   
   9.2. [Improving the accuracy](#9.2)
   
10. [The workflow](#10)

    10.1. [Standard approach - use all the orbitals](#10.1)
    
    10.2. [Using the reduced approach - use only a sub-space of orbitals](#10.2)
    
    10.3. [Let's compare the time-overlaps computed in the two ways](#10.3)


### A. Learning objectives

* To generate the input files for MOPAC calculations from the xyz trajectory file
* To extract the key infromation from the output file of MOPAC calculations (energies, MOs, CI vectors, etc.)
* To automate the calculations of the CI time-overlaps, adiabatic Hamiltonians, and adiabatic vibronic Hamiltonians using MOPAC


### B. Use cases

* INDO calculations with Libra 
* manually construct a Slater Determinant bais 
* processing the MOPAC calculations results
* computing CI wavefunction time-overlaps with MOPAC
* define Libra/MOPAC interface Hamiltonian
* computing single-particle (KS-DFT, HF, semiempirical) NACs
* computing many-body (TD-DFT, TD-DFTB, CI) NACs

### C. Functions

- `libra_py`
  - `workflows`
    - `nbra`
      - `mapping2`
        - [`ovlp_mat_arb`](#ovlp_mat_arb-1)
  - `packages`
    - `cp2k`
      - `methods`
        - [`read_trajectory_xyz_file`](#read_trajectory_xyz_file-1) | [`also here`](#read_trajectory_xyz_file-2)
    - `mopac`
      - `methods`
        - [`make_mopac_input`](#make_mopac_input-1)
        - [`run_mopac`](#run_mopac-1)
        - [`make_ref`](#make_ref-1)
        - [`make_alpha_excitation`](#make_alpha_excitation-1)
        - [`read_mopac_orbital_info`](#read_mopac_orbital_info-1)
        - [` mopac_compute_adi`](#mopac_compute_adi-1)
        

## 1. Importing needed libraries <a name="1"></a>
[Back to TOC](#toc)


In [1]:
import numpy as np
import matplotlib.pyplot as plt

from liblibra_core import *
import libra_py.data_conv as data_conv
import libra_py.packages.cp2k.methods as cp2k
import libra_py.packages.mopac.methods as mopac
import libra_py.workflows.nbra.mapping2 as mapping2
import libra_py.workflows.nbra.mapping3 as mapping3
import libra_py.units as units

import libra_py.citools.slatdet as sd
import libra_py.citools.csf as csf
import libra_py.citools.interfaces as interfaces

%matplotlib inline 

<frozen importlib._bootstrap>:219: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:219: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:219: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:219: RuntimeWar

## 2. Reading  a snapshot from the xyz trajectory file<a name="2"></a>
[Back to TOC](#toc)

We read the first (zero-th) snapshot of the `TiO2-aligned.xyz` trajectory file

The function determines the names of the N atoms in the system and saves the x, y, and z coordinates to MATRIX(3N,1) format and converts the coordinates from Angstrom to Bohr

We can print out the atomic labels - 1 Ti and 2 O - this is a TiO2 cluster.

The execution of this function also creates the corresponding coordinate xyz file, named "coord-0.xyz". The numerical index corresponds to the index of the snapshot we are reading in
<a name="read_trajectory_xyz_file-1"></a>

In [2]:
labels, q = cp2k.read_trajectory_xyz_file("TiO2-aligned.xyz", 0)

print(labels)
q.show_matrix()

['Ti', 'O', 'O']
-0.0000000  
-0.0037794520  
0.70297807  
-0.0000000  
2.6399472   
-1.0563568  
-0.0000000  
-2.6286089  
-1.0487979  



Read more:

## 3. Creating input files for MOPAC calculations<a name="3"></a>
[Back to TOC](#toc)

We are now ready to use this information to create an input file for MOPAC calculations using `make_mopac_input`
<a name="make_mopac_input-1"></a>

In [5]:
mopac_input_filename = "input.mop"
mopac_run_params = "INDO C.I.=(6,3) CHARGE=0 RELSCF=0.000001 ALLVEC  WRTCONF=0.00  WRTCI=2"
mopac.make_mopac_input(mopac_input_filename, mopac_run_params, labels, q)

## 4. Running MOPAC calculations<a name="4"></a>
[Back to TOC](#toc)

To prepare the working folders, input files (from the xyz trajectory files) and to run the MOPAC calculations, we can use the `run_mopac` function:
<a name="run_mopac-1"></a>

In [7]:
params_ = { "labels": labels, "mopac_exe":"/home/alexvakimov/SOFTWARE/mopac/_build/mopac" }

for i in range(5):
    _, q = cp2k.read_trajectory_xyz_file("TiO2-aligned.xyz", i)
    params_.update( {"mopac_jobid":F"job{i}"} )
    mopac.run_mopac(q, params_)

sh: /home/alexvakimov/SOFTWARE/mopac/_build/mopac: No such file or directory
sh: /home/alexvakimov/SOFTWARE/mopac/_build/mopac: No such file or directory
sh: /home/alexvakimov/SOFTWARE/mopac/_build/mopac: No such file or directory
sh: /home/alexvakimov/SOFTWARE/mopac/_build/mopac: No such file or directory
sh: /home/alexvakimov/SOFTWARE/mopac/_build/mopac: No such file or directory


## 5. Reading molecular orbitals/energies and CI vectors<a name="4"></a>
[Back to TOC](#toc)

Now we are ready to parse the output files generated. This is done using `read_mopac_orbital_info` function:
<a name="read_mopac_orbital_info-1"></a>

In [17]:
params = { "filename": "mopac_wd/input_job0.out",
           "orbital_space": [6,7,8, 9,10,11],}
E0, MO0, E_CI0, CI0, configs0, configs_raw0, actual_orbital_space = mopac.read_mopac_orbital_info(params)

params = { "filename": "mopac_wd/input_job1.out",
           "orbital_space": [6,7,8, 9,10,11] }
E1, MO1, E_CI1, CI1, configs1, configs_raw1, actual_orbital_Bpace = mopac.read_mopac_orbital_info(params)


### 6. Creating spin-adapted configurations (detailed)<a name="6"></a>
[Back to TOC](#toc)

Now, have some basis of excited Slater determinants. However, this is not a complete set of such determinants needed to construct all spin-adapted configurations (also known as configuration state functions, CSFs). 

For the considered above CAS(6,3), with the active orbitals being 6,7,8,9,10, and 11, we can construct all the CSFs using the `citools` module:

In [18]:
dets = list(sd.generate_determinants_with_parity([6,7,8,9,10,11], 6))

print(dets)
print(len(dets))

[((6, -6, 7, -7, 8, -8), 1), ((6, -6, 7, -7, 8, 9), 1), ((6, -6, 7, -7, 8, -9), 1), ((6, -6, 7, -7, 8, 10), 1), ((6, -6, 7, -7, 8, -10), 1), ((6, -6, 7, -7, 8, 11), 1), ((6, -6, 7, -7, 8, -11), 1), ((6, -6, 7, -7, -8, 9), 1), ((6, -6, 7, -7, -8, -9), 1), ((6, -6, 7, -7, -8, 10), 1), ((6, -6, 7, -7, -8, -10), 1), ((6, -6, 7, -7, -8, 11), 1), ((6, -6, 7, -7, -8, -11), 1), ((6, -6, 7, -7, 9, -9), 1), ((6, -6, 7, -7, 9, 10), 1), ((6, -6, 7, -7, 9, -10), 1), ((6, -6, 7, -7, 9, 11), 1), ((6, -6, 7, -7, 9, -11), 1), ((6, -6, 7, -7, -9, 10), 1), ((6, -6, 7, -7, -9, -10), 1), ((6, -6, 7, -7, -9, 11), 1), ((6, -6, 7, -7, -9, -11), 1), ((6, -6, 7, -7, 10, -10), 1), ((6, -6, 7, -7, 10, 11), 1), ((6, -6, 7, -7, 10, -11), 1), ((6, -6, 7, -7, -10, 11), 1), ((6, -6, 7, -7, -10, -11), 1), ((6, -6, 7, -7, 11, -11), 1), ((6, -6, 7, 8, -8, 9), 1), ((6, -6, 7, 8, -8, -9), 1), ((6, -6, 7, 8, -8, 10), 1), ((6, -6, 7, 8, -8, -10), 1), ((6, -6, 7, 8, -8, 11), 1), ((6, -6, 7, 8, -8, -11), 1), ((6, -6, 7, 8, 9, 

This creates a list of all possible determinants, which is quite large. With a CAS of only (6,3), a list of 943 determinants is created. If the active space is expanded to (10,5), the list becomes more than 100,000 determinants long. In the general workflow, this list is filtered down to only the necessary determinants. However, when running calculations with only singlet excitations, the list of necessary determinants is quite small and it is much more efficient to generate this list directly, rather than filtering down from the full list of all possible determinants. This allows for the use of a much larger CAS than would be feasible using the general workflow.

In [19]:
all_confs = sd.generate_single_excitations([6,7,8,9,10,10],6)

print(list(all_confs))

[((6, -6, 7, -7, 8, -8), 1), ((-6, 7, -7, 8, -8, 9), 1), ((6, 7, -7, 8, -8, -9), 1), ((-6, 7, -7, 8, -8, 10), 1), ((6, 7, -7, 8, -8, -10), 1), ((-6, 7, -7, 8, -8, 10), 1), ((6, 7, -7, 8, -8, -10), 1), ((6, -6, -7, 8, -8, 9), 1), ((6, -6, 7, 8, -8, -9), 1), ((6, -6, -7, 8, -8, 10), 1), ((6, -6, 7, 8, -8, -10), 1), ((6, -6, -7, 8, -8, 10), 1), ((6, -6, 7, 8, -8, -10), 1), ((6, -6, 7, -7, -8, 9), 1), ((6, -6, 7, -7, 8, -9), 1), ((6, -6, 7, -7, -8, 10), 1), ((6, -6, 7, -7, 8, -10), 1), ((6, -6, 7, -7, -8, 10), 1), ((6, -6, 7, -7, 8, -10), 1)]


We can use the build_minimal_csf_basis_singlet function to make use of generate_single_excitations 

In [20]:
min_basis, (all_confs, all_phases) = interfaces.build_minimal_csf_basis_singlet(configs_raw1, active_space=[6,7,8,9,10,11], 
                                                                                nelec=6, max_unpaired=0)
print(min_basis)

[((6, -6, 7, -7, 8, -8), 1), ((-6, 7, -7, 8, -8, 9), 1), ((6, 7, -7, 8, -8, -9), 1), ((-6, 7, -7, 8, -8, 10), 1), ((6, 7, -7, 8, -8, -10), 1), ((-6, 7, -7, 8, -8, 11), 1), ((6, 7, -7, 8, -8, -11), 1), ((6, -6, -7, 8, -8, 9), 1), ((6, -6, 7, 8, -8, -9), 1), ((6, -6, -7, 8, -8, 10), 1), ((6, -6, 7, 8, -8, -10), 1), ((6, -6, -7, 8, -8, 11), 1), ((6, -6, 7, 8, -8, -11), 1), ((6, -6, 7, -7, -8, 9), 1), ((6, -6, 7, -7, 8, -9), 1), ((6, -6, 7, -7, -8, 10), 1), ((6, -6, 7, -7, 8, -10), 1), ((6, -6, 7, -7, -8, 11), 1), ((6, -6, 7, -7, 8, -11), 1)]


We now map the min_basis to be consistent with the reduced space of orbitals defined earlier and calculate the T matrix as normal

In [21]:

mapped = interfaces.map_to_active_indices(all_confs, active_space=[6,7,8,9,10,11])

for i in range( len(mapped) ):
    print(F"Configuration {i}: {all_confs[i]}  -> {mapped[i]}")   

Configuration 0: (6, -6, 7, -7, 8, -8)  -> (1, -1, 2, -2, 3, -3)
Configuration 1: (-6, 7, -7, 8, -8, 9)  -> (-1, 2, -2, 3, -3, 4)
Configuration 2: (6, 7, -7, 8, -8, -9)  -> (1, 2, -2, 3, -3, -4)
Configuration 3: (-6, 7, -7, 8, -8, 10)  -> (-1, 2, -2, 3, -3, 5)
Configuration 4: (6, 7, -7, 8, -8, -10)  -> (1, 2, -2, 3, -3, -5)
Configuration 5: (-6, 7, -7, 8, -8, 11)  -> (-1, 2, -2, 3, -3, 6)
Configuration 6: (6, 7, -7, 8, -8, -11)  -> (1, 2, -2, 3, -3, -6)
Configuration 7: (6, -6, -7, 8, -8, 9)  -> (1, -1, -2, 3, -3, 4)
Configuration 8: (6, -6, 7, 8, -8, -9)  -> (1, -1, 2, 3, -3, -4)
Configuration 9: (6, -6, -7, 8, -8, 10)  -> (1, -1, -2, 3, -3, 5)
Configuration 10: (6, -6, 7, 8, -8, -10)  -> (1, -1, 2, 3, -3, -5)
Configuration 11: (6, -6, -7, 8, -8, 11)  -> (1, -1, -2, 3, -3, 6)
Configuration 12: (6, -6, 7, 8, -8, -11)  -> (1, -1, 2, 3, -3, -6)
Configuration 13: (6, -6, 7, -7, -8, 9)  -> (1, -1, 2, -2, -3, 4)
Configuration 14: (6, -6, 7, -7, 8, -9)  -> (1, -1, 2, -2, 3, -4)
Configuratio

In [22]:
T = interfaces.conf2csf_matrix(min_basis, all_confs, all_phases, S = 0, Ms = 0)

## 7. Creating spin-adapted configurations (easier)<a name="6"></a>
[Back to TOC](#toc)

Now lets look at a higher-level function that takes care of creation of CSFs. In the end, we only need two things for the further calculations:

- the basis of configurations with the indexing within the selected orbital space
- the SD-to-CSFs transformation coefficients (T)

These calculations can be streamlined like this:
The only difference between this and the general workflow is the name of the function - configs_and_T_matrix_singlet rather than configs_and_T_matrix. The parameters and outputs are the same.

In [26]:
mapped_basis0, T0 = interfaces.configs_and_T_matrix_singlet(configs_raw0, 
                                                    active_space=[6,7,8,9,10,11], 
                                                    orbital_space=[6,7,8,9,10,11],
                                                    nelec=6, S=0, Ms=0)
mapped_basis1, T1 = interfaces.configs_and_T_matrix_singlet(configs_raw1, 
                                                    active_space=[6,7,8,9,10,11], 
                                                    orbital_space=[6,7,8,9,10,11],
                                                    nelec=6, S=0, Ms=0)

## 8.Computing time-overlaps<a name="6"></a>
[Back to TOC](#toc)

Finally, we can use the information read from the two timesteps to compute some properties.

The following calculations are the same as the general workflow

In [27]:
nmo = MO0.num_of_cols
print(nmo)

mo_st = MO0.T() * MO1  # time-overlap in MO basis
mo_st.show_matrix()

# First, let's convert the MO overlaps to numpy format
mo_st_np = data_conv.MATRIX2nparray(mo_st).real  # MATRIX -> real np array
print(mo_st_np)

# Next, compute the matrix of SD overlaps
time_ovlp_sd_np = sd.slater_overlap_matrix(mapped_basis0, mapped_basis1, mo_st_np)

# Convert them back
time_ovlp_sd = data_conv.nparray2MATRIX( time_ovlp_sd_np )  # complex np array -> MATRIX

print("Time-overlap in the CSF basis")
time_ovlp_csf = T0.real().T() * time_ovlp_sd * T1.real()
time_ovlp_csf.show_matrix()

print("Time-overlap in the CI basis")
time_ovlp_ci = CI0.T() * time_ovlp_csf * CI1
time_ovlp_ci.show_matrix()

6
[[ 9.99841725e-01  0.00000000e+00  4.50816370e-03 -7.39430000e-06
   0.00000000e+00 -7.26678900e-04]
 [ 0.00000000e+00  9.99157141e-01  0.00000000e+00  0.00000000e+00
  -7.48165600e-04  0.00000000e+00]
 [-4.52651650e-03  0.00000000e+00  9.99961678e-01 -7.03880900e-04
   0.00000000e+00 -4.17401000e-04]
 [-2.71870000e-06  0.00000000e+00  7.01338100e-04  1.00000101e+00
   0.00000000e+00  7.66569400e-04]
 [ 0.00000000e+00  7.60193400e-04  0.00000000e+00  0.00000000e+00
   9.99986591e-01  0.00000000e+00]
 [ 7.30125200e-04  0.00000000e+00  4.19473800e-04 -7.69201300e-04
   0.00000000e+00  9.99990298e-01]]
0.99984172  0.0000000   0.0045081637  -7.3943000e-06  0.0000000   -0.00072667890  
0.0000000   0.99915714  0.0000000   0.0000000   -0.00074816560  0.0000000   
-0.0045265165  0.0000000   0.99996168  -0.00070388090  0.0000000   -0.00041740100  
-2.7187000e-06  0.0000000   0.00070133810  1.0000010   0.0000000   0.00076656940  
0.0000000   0.00076019340  0.0000000   0.0000000   0.99998659  0

## 9. Overlaps
Now, let's apply these transformations to normal overlaps as well. The calculations are also the same as the general workflow.

In [28]:
print("Overlaps in the MO basis")
mo_s = MO0.T() * MO0  # time-overlap in MO basis
mo_s.show_matrix()

print("Overlaps in the SD basis")
# Next, compute the matrix of SD overlaps

mo_s_np = data_conv.MATRIX2nparray(mo_s).real  # MATRIX -> real np array
ovlp_sd_np = sd.slater_overlap_matrix(mapped_basis0, mapped_basis0, mo_s_np)
ovlp_sd = data_conv.nparray2MATRIX( ovlp_sd_np )  # complex np array -> MATRIX
ovlp_sd.show_matrix()

print("Overlaps in the CSF basis")
ovlp_csf = T0.real().T() * ovlp_sd * T0.real()
ovlp_csf.show_matrix()

print("Overlap in the CI basis")
ovlp_ci = CI0.T() * ovlp_csf * CI0
ovlp_ci.show_matrix()

Overlaps in the MO basis
Overlaps in the SD basis
0.99999992  0.0000000   -6.1874000e-06  -3.5278000e-06  0.0000000   6.4688000e-06  
0.0000000   0.99999716  0.0000000   0.0000000   -5.2327000e-06  0.0000000   
-6.1874000e-06  0.0000000   1.0000125   2.5236000e-06  0.0000000   9.0000000e-09  
-3.5278000e-06  0.0000000   2.5236000e-06  1.0000036   0.0000000   2.9709000e-06  
0.0000000   -5.2327000e-06  0.0000000   0.0000000   0.99999665  0.0000000   
6.4688000e-06  0.0000000   9.0000000e-09  2.9709000e-06  0.0000000   0.99999085  

Overlaps in the CSF basis
Overlap in the CI basis
1.0000191   -3.5278522e-06  -3.5278522e-06  0.0000000   0.0000000   6.4689244e-06  6.4689244e-06  0.0000000   0.0000000   5.2328150e-06  5.2328150e-06  0.0000000   0.0000000   2.5235949e-06  2.5235949e-06  0.0000000   0.0000000   9.0400851e-09  9.0400851e-09  
-3.5278522e-06  1.0000228   1.2445503e-11  0.0000000   -0.0000000  2.9709571e-06  -2.2820972e-11  0.0000000   -0.0000000  -1.8460245e-11  -1.8460245e-11

## 10.The workflow<a name="10"></a>
[Back to TOC](#toc)

Now, to automate the calculation of time-overlaps, vibronic and electronic Hamiltonians in the CI basis, we use the `mopac_compute_adi` function. We use the is_singlet_excitation parameter to use the singlet excitation specific functions
<a name="mopac_compute_adi-1"></a>

In [ ]:
%%time
wd = "tio2_wd"
labels, q = cp2k.read_trajectory_xyz_file("TiO2-aligned.xyz", 0)
params_elem = {"labels":labels, "timestep":0, "is_first_time":True,
               "CAS":[[6,7,8,9,10,11], 6],
               "orbital_space":[6,7,8,9,10,11],
               "mult_S":0, "mult_Ms":0,
               "is_singlet_excitations":True,
               "mopac_exe":"/home/alexvakimov/SOFTWARE/mopac/_build/mopac", 
               "mopac_run_params":"INDO C.I.=(6,3) CHARGE=0 RELSCF=0.000001 ALLVEC  WRTCONF=0.00  WRTCI=5",
               "mopac_working_directory":wd,
               "mopac_input_prefix":"input_", "mopac_output_prefix":"output_",
               "dt":1.0*units.fs2au
              }

# For 1 trajectory
params = [ dict(params_elem)]

# Emulates 1 trajectory
full_id = Py2Cpp_int([0, 0])

# Do the first 5 steps 
for i in range(5):
    labels, q = cp2k.read_trajectory_xyz_file("TiO2-aligned.xyz", i)
    params[0]["timestep"] = i
    
    obj = mopac.mopac_compute_adi(q, params, full_id)        
    obj.ham_adi.show_matrix(F"{wd}/ham_adi_{i}.txt")
    obj.hvib_adi.show_matrix(F"{wd}/hvib_adi_{i}.txt")
    obj.time_overlap_adi.real().show_matrix(F"{wd}/st_adi_{i}.txt")